Данные:
Фондовые индексы: 
- CША: https://uk.investing.com/indices/us-30-historical-data (DJI)
- Великобритания: https://www.wsj.com/market-data/quotes/index/UK/UKX/historical-prices (FTSE 100)
- Германия: https://stooq.com/q/d/?f=19920101&t=20000425&s=%5Edax&c=0 (DAX)


Доходности долгосрочных облигаций:
- США: https://fred.stlouisfed.org/series/DGS10 (DGS10)
- Великобритания: https://www.bankofengland.co.uk/statistics/yield-curves (uk_10y_full)
- Германия: 

Дальше код для подготовки данных, для каждой страны:
1. Фондовые индексы объединяются с доходностями облигаций за соответвующие периоды
2. В процессе все колонки приводятся к стандартному виду (даты, названия колонок)
3. Считаются вспомогательные для рассчетов колонки (логарифмическая разность, s, i, Y, ER)
4. Дальше эти данные будут анализированы в другом ноутбуке

In [32]:
%pip install pandas
%pip install numpy
%pip install matplotlib
%pip install statsmodels
%pip install scipy
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/27.9 MB ? eta -:--:--
   --- ------------------------------------ 2.4/27.9 MB 19.7 MB/s eta 0:00:02
   ------------ --------------------------- 8.7/27.9 MB 27.0 MB/s eta 0:00:01
   ---------------------- ----------------- 15.5/27.9 MB 29.6 MB/s eta 0:00:01
   --------------------------------- ------ 23.1/27.9 MB 31.9 MB/s eta 0:00:01
   ---------------------------------------- 27.9/27.9 MB 31.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Временные периоды (авторы делят таблицы на panel a, panel b, panel c, которые соответвуют всему периоду, докризисному и кризисному)

А также пути к csv файлам (для некоторых стран нет фьючерсов, ниже будет причина), и название папки, в которую сохранятся обработанные parquet файлы.

In [33]:
import pandas as pd
import numpy as np

TIME_PERIODS = {
    "full_time":["1992-01-03", "2000-04-20"],
    "sub1":["1992-01-03", "1997-05-30"],
    "sub2":["1997-06-02", "2000-04-20"]
    }

CSV_PATHS = {
    "us_indexes":r"raw_csv\US\dji.csv",
    "us_bonds":r"raw_csv\US\DGS10.csv",
    "us_futures":"",

    "uk_indexes":r"raw_csv\UK\FTSE.csv",
    "uk_bonds":r"raw_csv\UK\uk_10y_full.csv",
    "uk_futures":"",

    "bd_indexes":r"raw_csv\BD\dax.csv",
    "bd_bonds":r"raw_csv\BD\de10y_stooq.csv",
    "bd_futures":"",
}

OUTPUT_DIR = "data_for_analysis"

Функция для парсинга csv'шных данных если там строковые значения

In [3]:
def process_string_csv_data(csv_path):
    out_df = pd.read_csv(
        csv_path,
        header=0,
        names=["Date", "Price", "Open", "High", "Low", "Vol", "Change_%"],
    )

    out_df["Date"] = pd.to_datetime(out_df["Date"], format="%d/%m/%Y")

    for col in out_df.columns[1:]:
        out_df[col] = (
            out_df[col]
            .astype(str)
            .str.replace(",", "", regex=False)  
            .str.replace("%", "", regex=False)  
            .str.replace("M", "", regex=False)  
            .str.strip()
        )
        out_df[col] = pd.to_numeric(out_df[col], errors="coerce")
    return out_df


Пойдем по странам, начинаем с фондового индекса США: DJI

In [17]:
dji = process_string_csv_data(CSV_PATHS["us_indexes"])
dji_prices = dji[["Date", "Price"]]
print(dji.info())
print('\n-------------------\n')
print(dji_prices.describe())

<class 'pandas.DataFrame'>
RangeIndex: 2681 entries, 0 to 2680
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   Date      2681 non-null   datetime64[us]
 1   Price     2681 non-null   float64       
 2   Open      2681 non-null   float64       
 3   High      2681 non-null   float64       
 4   Low       2681 non-null   float64       
 5   Vol       2101 non-null   float64       
 6   Change_%  2681 non-null   float64       
dtypes: datetime64[us](1), float64(6)
memory usage: 146.7 KB
None

-------------------

                             Date         Price
count                        2681   2681.000000
mean   1996-06-19 08:25:57.627750   6420.434946
min           1992-01-01 00:00:00   3136.580000
25%           1994-07-22 00:00:00   3801.460000
50%           1996-08-24 00:00:00   5719.270000
75%           1998-06-25 00:00:00   8775.390000
max           2000-04-25 00:00:00  11722.980000
std                

Дальше доходности облигаций, приводим к нужному формату и склеиваем их с датафреймом фондового индекса по датам

In [18]:
dgs = pd.read_csv(CSV_PATHS["us_bonds"])
dgs["observation_date"] = pd.to_datetime(dgs["observation_date"], format="%Y-%m-%d")
dgs = dgs.rename(columns={"observation_date":"Date", "DGS10":"BondYield"})
print(dgs.info())
print(dgs.describe())

<class 'pandas.DataFrame'>
RangeIndex: 2169 entries, 0 to 2168
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       2169 non-null   datetime64[us]
 1   BondYield  2082 non-null   float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 34.0 KB
None
                      Date    BondYield
count                 2169  2082.000000
mean   1996-02-28 00:00:00     6.283079
min    1992-01-02 00:00:00     4.160000
25%    1994-01-31 00:00:00     5.770000
50%    1996-02-28 00:00:00     6.250000
75%    1998-03-27 00:00:00     6.790000
max    2000-04-25 00:00:00     8.050000
std                    NaN     0.753005


Объединяем по дате

In [19]:
df_us = pd.merge(dji_prices, dgs, on="Date", how="inner")
df_us.head()

,Date,Price,BondYield
0,2000-04-25,11124.83,6.14
1,2000-04-24,10906.10,6.00
2,2000-04-21,10844.06,NaN
3,2000-04-20,10844.06,5.99
4,2000-04-19,10674.97,5.99


Смотрим сколько пропусков. Их довольно мало, но удалять не хочется, чтобы сохранить побольше дат, заполняем пропуски предыдущими значениями в ячейках

In [20]:
def check_nans(df):
    print("Количество пропусков:")
    print(df.isna().sum())

    print("\nПроцент пропусков:")
    print((df.isna().mean() * 100).round(2))

    total_nans = df.isna().sum().sum()
    print(f"\nВсего NaN в таблице: {total_nans}")

check_nans(df_us)
df_us_interp = df_us.ffill()
check_nans(df_us_interp)

Количество пропусков:
Date          0
Price         0
BondYield    87
dtype: int64

Процент пропусков:
Date         0.00
Price        0.00
BondYield    4.01
dtype: float64

Всего NaN в таблице: 87
Количество пропусков:
Date         0
Price        0
BondYield    0
dtype: int64

Процент пропусков:
Date         0.0
Price        0.0
BondYield    0.0
dtype: float64

Всего NaN в таблице: 0


### Дальше расчет вспомогательных столбцов

Есть некоторые проблемы с доступом к данным по фьючерсам (в открытых источниках нет данных по 1992-2000 годам, а к закрытым у нас нет доступа)

Поэтому можем попробовать альтернативный способ(по сути это небольшое упрощение по сравнению со статьей, поэтому результаты могут немного отличаться): вместо дневного изменения цены фьючерса на облигацию (как делают авторы) использовать приближение/упрощение - рассчитать изменение цены на облигацию через изменение доходности и модифицированную дюрацию

$$\frac{\Delta P}{P} \approx -D_{mod} \cdot \Delta i$$

Где $D_{mod}$ - модифицированная дюрация, $\Delta i$ - изменение доходности, а $\frac{\Delta P}{P}$ - изменение рыночной цены на облигацию

Изменение доходности $\Delta i$ мы имеем в DGS10 (и аналогах для других рынков), дюрацию же можно взять равной $7.5$ для упрощения

Берем такое значение по следующим причинам:
Для 10-ти летней купонной облигации с доходностью к погашению примерно равной 5.5%-6% дюрация Маколея примерна равна 8 лет

Тогда модифицированную дюрацию можно посчитать как: $D_{mod} = \frac{D_{mac}}{1 + \frac{y}{m}} $, где $y \approx 0.06$ (средняя ставка доходности 10-летних US Treasuries в 1990-х годах), $m = 2$ (полугодовые купонные выплаты)

При таком упрощении получаем $D_{mod} \approx 7.5$ 

Опять таки, из-за этого упрощения конкретные полученные цифры могут расходиться со статьей, но, как будет наглядно видно далее, расхождения в пределах допустимого и не мешают получению таких же выводов какие были получены в оригинальной статье

In [21]:
def process_colums(df, country_prefix: str, include_duration_method=True):
    df = df.sort_values("Date").reset_index(drop=True).copy()
    df[f"s_{country_prefix}"] = 100 * (np.log(df["Price"]) - np.log(df["Price"].shift(1)))
    df[f"i_{country_prefix}"] = df["BondYield"] / 365.0
    df[f"delta_i_{country_prefix}"] = df["BondYield"] - df["BondYield"].shift(1)
    if include_duration_method:
        df[f"Y_{country_prefix}"] = -7.5 * df[f"delta_i_{country_prefix}"]
    else:
        pass
    df[f"ER_{country_prefix}"] = df[f"s_{country_prefix}"] - df[f"i_{country_prefix}"]
    return df

In [22]:
df_us_processed = process_colums(df_us_interp, "US")
df_us_processed = df_us_processed.dropna()
print(df_us_processed.head(10))
print("\n------------------\n")
print(df_us_processed.describe())


         Date    Price  BondYield      s_US      i_US  delta_i_US   Y_US  \
1  1992-01-03  3201.47       6.85  0.911853  0.018767        0.07 -0.525   
2  1992-01-06  3200.13       6.82 -0.041865  0.018685       -0.03  0.225   
3  1992-01-07  3204.83       6.76  0.146761  0.018521       -0.06  0.450   
4  1992-01-08  3203.93       6.77 -0.028087  0.018548        0.01 -0.075   
5  1992-01-09  3209.53       6.79  0.174633  0.018603        0.02 -0.150   
6  1992-01-10  3199.46       6.85 -0.314246  0.018767        0.06 -0.450   
7  1992-01-13  3185.60       6.92 -0.434139  0.018959        0.07 -0.525   
8  1992-01-14  3246.20       7.03  1.884443  0.019260        0.11 -0.825   
9  1992-01-15  3258.50       7.05  0.378189  0.019315        0.02 -0.150   
10 1992-01-16  3249.55       7.13 -0.275044  0.019534        0.08 -0.600   

       ER_US  
1   0.893086  
2  -0.060549  
3   0.128241  
4  -0.046635  
5   0.156030  
6  -0.333013  
7  -0.453098  
8   1.865182  
9   0.358873  
10 -0.294578 

Эти данные уже готовы к анализу для таблицы 1 и следующих из статьи, повторим процесс для других стран

Для Великобритании:

In [23]:
ftse = pd.read_csv(CSV_PATHS["uk_indexes"])
ftse = ftse.rename(columns={" Close":"Price"})
ftse_prices = ftse[["Date", "Price"]]
ftse_prices["Date"] = pd.to_datetime(ftse_prices["Date"], format="%m/%d/%y")
ftse_prices.info()

<class 'pandas.DataFrame'>
RangeIndex: 2101 entries, 0 to 2100
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    2101 non-null   datetime64[us]
 1   Price   2101 non-null   float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 33.0 KB


In [24]:
uk_10y = pd.read_csv(CSV_PATHS["uk_bonds"])
uk_10y = uk_10y.rename(columns={"date":"Date", "yield_10.0y":"BondYield"})
uk_10y["Date"] = pd.to_datetime(uk_10y["Date"], format="%Y-%m-%d")
df_uk = pd.merge(ftse_prices, uk_10y, on="Date", how="inner")
df_uk.head()

,Date,Price,BondYield
0,2000-04-25,6282.97,5.235517
1,2000-04-20,6241.22,5.192162
2,2000-04-19,6184.91,5.184162
3,2000-04-18,6074.04,5.176914
4,2000-04-17,5994.57,5.195414


In [25]:
df_uk_processed = process_colums(df_uk, "UK")
df_uk_processed = df_uk_processed.dropna()
df_uk_processed.describe()

,Date,Price,BondYield,s_UK,i_UK,delta_i_UK,Y_UK,ER_UK
count,2100,2100.000000,2100.000000,2100.000000,2100.000000,2100.000000,2100.000000,2100.000000
mean,1996-02-26 14:47:18.857142,4142.672095,7.171130,0.044021,0.019647,-0.001913,0.014344,0.024374
min,1992-01-03 00:00:00,2281.000000,4.122027,-4.139903,0.011293,-0.495470,-2.582430,-4.165363
25%,1994-01-30 06:00:00,3039.600000,5.810190,-0.504159,0.015918,-0.038281,-0.237947,-0.524240
50%,1996-02-26 12:00:00,3707.850000,7.574415,0.050252,0.020752,-0.003583,0.026876,0.031012
75%,1998-03-24 06:00:00,5330.900000,8.307004,0.583603,0.022759,0.031726,0.287104,0.561716
max,2000-04-25 00:00:00,6930.200000,9.653103,5.439552,0.026447,0.344324,3.716027,5.415035
std,NaN,1330.200624,1.396478,0.945754,0.003826,0.063909,0.479315,0.945809


Далее Германия

In [27]:
dax = pd.read_csv(CSV_PATHS["bd_indexes"])
dax = dax.rename(columns={"Close":"Price"})
dax["Date"] = pd.to_datetime(dax["Date"], format="%Y-%m-%d")
dax_prices = dax[["Date", "Price"]]
dax_prices.describe()

,Date,Price
count,2089,2089.000000
mean,1996-02-27 15:44:22.517951,3211.733389
min,1992-01-02 00:00:00,1420.300000
25%,1994-01-28 00:00:00,2027.400000
50%,1996-02-23 00:00:00,2444.900000
75%,1998-03-30 00:00:00,4595.820000
max,2000-04-25 00:00:00,8064.970000
std,NaN,1615.600879


In [30]:
de_10y = pd.read_csv(CSV_PATHS["bd_bonds"])
de_10y = de_10y.rename(columns={"date":"Date", "close":"BondYield"})
de_10y["Date"] = pd.to_datetime(de_10y["Date"], format="%Y-%m-%d")
df_bd = pd.merge(dax_prices, de_10y, on="Date", how="inner")
df_bd.head()

,Date,Price,BondYield
0,1992-01-02,1601.9,8.043
1,1992-01-03,1603.6,8.054
2,1992-01-06,1603.3,8.010
3,1992-01-07,1592.5,7.989
4,1992-01-08,1578.7,8.005


In [31]:
df_bd_processed = process_colums(df_bd, "BD")
df_bd_processed = df_bd_processed.dropna()
df_bd_processed.describe()

,Date,Price,BondYield,s_BD,i_BD,delta_i_BD,Y_BD,ER_BD
count,2068,2068.000000,2068.000000,2068.000000,2068.000000,2068.000000,2068.000000,2068.000000
mean,1996-02-27 05:26:34.584139,3211.680575,6.077014,0.073211,0.016649,-0.001309,0.009814,0.056562
min,1992-01-03 00:00:00,1420.300000,3.629000,-7.241084,0.009942,-0.278000,-2.662500,-7.252156
25%,1994-01-23 06:00:00,2026.625000,5.292250,-0.523261,0.014499,-0.028000,-0.165000,-0.538366
50%,1996-02-19 12:00:00,2435.900000,6.081000,0.103433,0.016660,0.000000,-0.000000,0.087277
75%,1998-03-31 06:00:00,4600.085000,6.864500,0.763024,0.018807,0.022000,0.210000,0.746046
max,2000-04-25 00:00:00,8064.970000,8.167000,5.893675,0.022375,0.355000,2.085000,5.882091
std,NaN,1618.374972,1.132168,1.228743,0.003102,0.049886,0.374141,1.228829


Теперь разобьем по нужным периодам и запишем в файлы, по которым уже будем проводить анализ

In [ ]:
import os

us_processed_full_time = df_us_processed[(df_us_processed["Date"] >= TIME_PERIODS["full_time"][0]) & (df_us_processed["Date"] <= TIME_PERIODS["full_time"][1])].copy()
us_processed_sub1 = df_us_processed[(df_us_processed["Date"] >= TIME_PERIODS["sub1"][0]) & (df_us_processed["Date"] <= TIME_PERIODS["sub1"][1])].copy()
us_processed_sub2 = df_us_processed[(df_us_processed["Date"] >= TIME_PERIODS["sub2"][0]) & (df_us_processed["Date"] <= TIME_PERIODS["sub2"][1])].copy()

uk_processed_full_time = df_uk_processed[(df_uk_processed["Date"] >= TIME_PERIODS["full_time"][0]) & (df_uk_processed["Date"] <= TIME_PERIODS["full_time"][1])].copy()
uk_processed_sub1 = df_uk_processed[(df_uk_processed["Date"] >= TIME_PERIODS["sub1"][0]) & (df_uk_processed["Date"] <= TIME_PERIODS["sub1"][1])].copy()
uk_processed_sub2 = df_uk_processed[(df_uk_processed["Date"] >= TIME_PERIODS["sub2"][0]) & (df_uk_processed["Date"] <= TIME_PERIODS["sub2"][1])].copy()

bd_processed_full_time = df_bd_processed[(df_bd_processed["Date"] >= TIME_PERIODS["full_time"][0]) & (df_bd_processed["Date"] <= TIME_PERIODS["full_time"][1])].copy()
bd_processed_sub1 = df_bd_processed[(df_bd_processed["Date"] >= TIME_PERIODS["sub1"][0]) & (df_bd_processed["Date"] <= TIME_PERIODS["sub1"][1])].copy()
bd_processed_sub2 = df_bd_processed[(df_bd_processed["Date"] >= TIME_PERIODS["sub2"][0]) & (df_bd_processed["Date"] <= TIME_PERIODS["sub2"][1])].copy()

os.makedirs(OUTPUT_DIR, exist_ok=True)

datasets = {
    "us_processed_full_time.parquet": us_processed_full_time,
    "us_processed_sub1.parquet": us_processed_sub1,
    "us_processed_sub2.parquet": us_processed_sub2,
}

for name, dataframe in datasets.items():
    path = os.path.join(OUTPUT_DIR, name)
    dataframe.to_parquet(path, index=False)





ArrowKeyError: No type extension with name arrow.py_extension_type found